# Checking a submission

In this notebook we run `check_submission`, the function behind the `check-submission` command, on
two of the baselines in `baselines/`, and we look at how the track of each is computed from its
controls. 

In [1]:
import shutil
import sys
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from starterkit import BASELINES_DIR, BASE_MODELS, METHOD_CARD, REPO_ROOT, model_slug
from starterkit.baselines import fetch_baselines
from starterkit.check import check_submission

fetch_baselines()          # downloads the baselines from the pinned Hub revision

print("base models:", BASE_MODELS)
print("baselines:  ", sorted(p.name for p in BASELINES_DIR.iterdir()
                             if p.is_dir() and not p.name.startswith(".")))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

base models: ('Qwen/Qwen3.8-27B', 'google/gemma-4-31B-it', 'ibm-granite/granite-4.2-30b')
baselines:   ['honest_persona', 'honesty_few_shot', 'honesty_sft', 'unsteered']


## Checks on two baselines

We start with `honesty_few_shot`, whose controls are a system prompt and a few-shot block that change only the prompt, which places it in the black-box
track.

In [2]:
report = check_submission(BASELINES_DIR / "honesty_few_shot")
assert report["ok"], report["errors"]
assert report["track"] == "black-box", report["track"]

print("ok:", report["ok"], "| track:", report["track"])
for model, record in report["bundles"].items():
    print(f"  {model}")
    print(f"    {record['file']}  {record['size_bytes']:,} bytes")
    print(f"    methods: {record['methods']}  ->  {record['track']}")

ok: True | track: black-box
  Qwen/Qwen3.8-27B
    Qwen__Qwen3.8-27B.spipe  674,785 bytes
    methods: ['input_control/system_prompt', 'input_control/few_shot']  ->  black-box
  google/gemma-4-31B-it
    google__gemma-4-31B-it.spipe  673,768 bytes
    methods: ['input_control/system_prompt', 'input_control/few_shot']  ->  black-box
  ibm-granite/granite-4.2-30b
    ibm-granite__granite-4.2-30b.spipe  674,796 bytes
    methods: ['input_control/system_prompt', 'input_control/few_shot']  ->  black-box


The `honesty_sft` pipeline contains a `load_lora` entry that changes the model's weights which places it in the open track.

In [3]:
report = check_submission(BASELINES_DIR / "honesty_sft")
assert report["ok"], report["errors"]
assert report["track"] == "open", report["track"]

print("ok:", report["ok"], "| track:", report["track"])
for model, record in report["bundles"].items():
    print(f"  {record['file']:36s} {record['size_bytes']:>13,} bytes  {record['methods']}")

# each bundle contains its own LoRA adapter, and each is tens of MB
print("\ntotal:", sum(r["size_bytes"] for r in report["bundles"].values()), "bytes")

ok: True | track: open
  Qwen__Qwen3.8-27B.spipe                 44,146,674 bytes  ['structural_control/sft']
  google__gemma-4-31B-it.spipe           119,900,849 bytes  ['structural_control/sft']
  ibm-granite__granite-4.2-30b.spipe      61,744,737 bytes  ['structural_control/sft']

total: 225792260 bytes


## Track detection

The checker computes the track from the controls' declarations. For each control it reads the
toolkit base class, `steer_access()`, and the backend capabilities `requirements()` lists, and takes
the highest implied track. A bundle's track is the highest over its enabled controls. A custom control inherits all three declarations
from the base class it subclasses.

The cell below prints each control's track and the underlying reason. Note that `honest_persona` uses a
`PhasedDecoding` driver so it returns black-box, while `honesty_sft` contains a `load_lora` entry and
returns open.

In [4]:
from steerability.spipe import SPipe

from starterkit import BACKEND, PIPELINE_KWARGS
from starterkit.track import control_tracks, track_of

slug = model_slug(BASE_MODELS[0])
for name in ("honest_persona", "honesty_sft"):
    path = BASELINES_DIR / name / f"{slug}.spipe"
    pipeline = SPipe.load(path, allow_code=True).pipeline(backend=BACKEND, **PIPELINE_KWARGS)
    controls = (*pipeline.structural_controls, *pipeline.input_controls,
                *pipeline.state_controls, *pipeline.output_controls)
    print(f"{name}  ->  {track_of(controls)}")
    for entry in control_tracks(controls):
        flag = "" if entry["enabled"] else "  [disabled, not counted]"
        print(f"    {entry['method']:44s} {entry['track']:10s} ({entry['reason']}){flag}")
    print()

honest_persona  ->  black-box
    input_control/system_prompt                  black-box  (prompt or generation-level only)
    input_control/user_prefix                    black-box  (prompt or generation-level only)
    output_control/phased_decoding               black-box  (prompt or generation-level only)

honesty_sft  ->  open
    structural_control/load_lora                 open       (structural control)



The `starterkit.track.ATOM_TRACK` maps each of the toolkit's `Capability` atoms to a track (printed below).

In [5]:
from starterkit.track import ATOM_TRACK

for atom, track in sorted(ATOM_TRACK.items(), key=lambda item: (item[1], item[0].value)):
    print(f"  {atom.value:24s} {track}")

  beam_proposals           black-box
  serve_checkpoint         open
  serve_lora               open
  guided_decoding          white-box
  hidden_capture           white-box
  in_process_torch         white-box
  intervention_specs       white-box
  per_step_logit_specs     white-box


## Failure cases

We build a few submissions with one defect each in a temporary directory, leaving `baselines/` as it is, and print the errors the checker reports for each. These are the errors our submission intake flags. The cases are:

- (a) a bundle missing for one model
- (b) bundles with different control sequences
- (c) a missing method card
- (d) a method card with no content
- (e) a bundle not named `<model-slug>.spipe`


In [6]:
def show(label, directory):
    report = check_submission(directory)
    print(f"--- {label}")
    print(f"    ok={report['ok']}  track={report['track']}")
    for error in report["errors"]:
        print(f"    error: {error}")
    for warning in report["warnings"]:
        print(f"    warning: {warning}")
    return report


workdir = Path(tempfile.mkdtemp(prefix="failure-cases-"))


# (a) a bundle missing for one model
case = workdir / "missing-model"
shutil.copytree(BASELINES_DIR / "honesty_few_shot", case)
(case / f"{model_slug(BASE_MODELS[1])}.spipe").unlink()
show("(a) a bundle missing for one model", case)


# (b) bundles with different control sequences; honesty_sft for one model and
#     honesty_few_shot for the others
case = workdir / "mixed-methods"
case.mkdir()
shutil.copyfile(BASELINES_DIR / "honesty_few_shot" / METHOD_CARD, case / METHOD_CARD)
shutil.copyfile(BASELINES_DIR / "honesty_sft" / f"{model_slug(BASE_MODELS[0])}.spipe",
                case / f"{model_slug(BASE_MODELS[0])}.spipe")
for model in BASE_MODELS[1:]:
    shutil.copyfile(BASELINES_DIR / "honesty_few_shot" / f"{model_slug(model)}.spipe",
                    case / f"{model_slug(model)}.spipe")
# both errors are reported, since the sequences differ and the tracks differ (open and black-box)
show("(b) bundles with different control sequences", case)


# (c) a missing method card
case = workdir / "no-card"
shutil.copytree(BASELINES_DIR / "honesty_few_shot", case)
(case / METHOD_CARD).unlink()
show("(c) a missing method card", case)


# (d) a method card with no content
case = workdir / "empty-card"
shutil.copytree(BASELINES_DIR / "honesty_few_shot", case)
(case / METHOD_CARD).write_text("\n", encoding="utf-8")
show("(d) a method card with no content", case)


# (e) a bundle not named <model-slug>.spipe
case = workdir / "misnamed-bundle"
shutil.copytree(BASELINES_DIR / "honesty_few_shot", case)
slug = model_slug(BASE_MODELS[0])
(case / f"{slug}.spipe").rename(case / "qwen.spipe")
show("(e) a bundle not named <model-slug>.spipe", case)


--- (a) a bundle missing for one model
    ok=False  track=black-box
    error: no bundle for google/gemma-4-31B-it; a submission covers every base model.


--- (b) bundles with different control sequences
    ok=False  track=None
    error: the bundles do not share one control sequence; the same method must be submitted for every base model (constructor arguments may differ per model). Qwen__Qwen3.8-27B.spipe: ['structural_control/sft']; google__gemma-4-31B-it.spipe: ['input_control/system_prompt', 'input_control/few_shot']; ibm-granite__granite-4.2-30b.spipe: ['input_control/system_prompt', 'input_control/few_shot']
    error: the bundles are not in one track: Qwen__Qwen3.8-27B.spipe: open; google__gemma-4-31B-it.spipe: black-box; ibm-granite__granite-4.2-30b.spipe: black-box
--- (c) a missing method card
    ok=False  track=black-box
    error: no method_card.md; every submission contains one.
--- (d) a method card with no content
    ok=False  track=black-box
    error: method_card.md is empty; every submission contains one with content.
--- (e) a bundle not named <model-slug>.spipe
    ok=False  track=black-box
    error: qwen.spipe: 

{'submission': '/tmp/failure-cases-sl4oy155/misnamed-bundle',
 'ok': False,
 'track': 'black-box',
 'errors': ['qwen.spipe: targets Qwen/Qwen3.8-27B; the bundle for that model must be named Qwen__Qwen3.8-27B.spipe.'],
 'warnings': [],
 'bundles': {'google/gemma-4-31B-it': {'file': 'google__gemma-4-31B-it.spipe',
   'size_bytes': 673768,
   'methods': ['input_control/system_prompt', 'input_control/few_shot'],
   'track': 'black-box',
   'control_tracks': [{'method': 'input_control/system_prompt',
     'enabled': True,
     'track': 'black-box',
     'reason': 'prompt or generation-level only'},
    {'method': 'input_control/few_shot',
     'enabled': True,
     'track': 'black-box',
     'reason': 'prompt or generation-level only'}]},
  'ibm-granite/granite-4.2-30b': {'file': 'ibm-granite__granite-4.2-30b.spipe',
   'size_bytes': 674796,
   'methods': ['input_control/system_prompt', 'input_control/few_shot'],
   'track': 'black-box',
   'control_tracks': [{'method': 'input_control/syste

## The submission archive

A passing submission is a directory with one bundle per base model, each named `<model-slug>.spipe`,
and the method card, optionally with a `controls/` package. We check `honesty_few_shot` once more
and print the command that zips it, since the archive to submit is the zipped directory.

In [7]:
submission = BASELINES_DIR / "honesty_few_shot"
report = check_submission(submission)
print("PASS" if report["ok"] else "FAIL", "| track:", report["track"])
print()
for path in sorted(submission.iterdir()):
    print(f"  {path.name:36s} {path.stat().st_size:>13,} bytes")
print()
print("The archive to submit:")
print(f"  cd {BASELINES_DIR.relative_to(REPO_ROOT)} && zip -r {submission.name}.zip {submission.name}")

PASS | track: black-box

  Qwen__Qwen3.8-27B.spipe                    674,785 bytes
  google__gemma-4-31B-it.spipe               673,768 bytes
  ibm-granite__granite-4.2-30b.spipe         674,796 bytes
  method_card.md                                 781 bytes

The archive to submit:
  cd baselines && zip -r honesty_few_shot.zip honesty_few_shot
